# Trois requêtes SPARQL sur les comptages

Exécuter les cellules dans l'ordre depuis Jupyter, après avoir produit Silver. Le graphe RDF contient l'ontologie et au plus 200 observations réelles de janvier 2025, sélectionnées dans Silver. Les résultats décrivent cet extrait et non tout le jeu de données.

In [ ]:
%pip install -q rdflib

In [ ]:
import sys
from pathlib import Path
from urllib.parse import quote

racine = Path.cwd()
if not (racine / 'commun').is_dir():
    racine = racine.parent
if not (racine / 'commun').is_dir():
    raise FileNotFoundError('Exécuter ce notebook depuis le dossier semantique du projet.')
sys.path.insert(0, str(racine))

from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD
from pyspark.sql import functions as F
from commun.spark_utils import LAKE, creer_session, format_tables

PARIS = Namespace('http://example.org/comptage-multimodal#')
g = Graph()
g.bind('', PARIS)
g.parse(str(racine / 'semantique' / 'ontologie_comptagesparis.ttl'), format='turtle')
spark = creer_session('SPARQL_Silver', lac=True, delta=True, postgres=False)
spark.conf.set('spark.sql.session.timeZone', 'Europe/Paris')
silver = spark.read.format(format_tables(spark)).load(f'{LAKE}/silver/comptage_multimodal_paris')
extrait = (silver.filter((F.col('date_heure') >= F.lit('2025-01-01').cast('timestamp'))
                         & (F.col('date_heure') < F.lit('2025-02-01').cast('timestamp'))
                         & F.col('nom_du_site').isNotNull())
    .select('identifiant_site', 'nom_du_site', 'mode_deplacement', 'nombre_vehicules',
            'type_voie', F.date_format('date_heure', "yyyy-MM-dd'T'HH:mm:ss").alias('date_locale'))
    .limit(200).collect())
if not extrait:
    raise ValueError('Aucune observation Silver avec nom de site en janvier 2025.')

def uri(type_ressource, valeur):
    return URIRef(f'{PARIS}{type_ressource}/{quote(str(valeur), safe="")}')

for index, ligne in enumerate(extrait):
    observation = uri('observation', index)
    site = uri('site', ligne.identifiant_site)
    mode = uri('mode', ligne.mode_deplacement)
    g.add((observation, RDF.type, PARIS.ObservationComptage))
    g.add((observation, PARIS.concerneSite, site))
    g.add((observation, PARIS.concerneMode, mode))
    g.add((observation, PARIS.dateHeureComptage, Literal(ligne.date_locale, datatype=XSD.dateTime)))
    g.add((observation, PARIS.nombreVehicules, Literal(ligne.nombre_vehicules, datatype=XSD.integer)))
    g.add((site, RDF.type, PARIS.SiteComptage))
    g.add((site, PARIS.identifiantSite, Literal(ligne.identifiant_site)))
    g.add((site, PARIS.nomSite, Literal(ligne.nom_du_site)))
    g.add((mode, RDF.type, PARIS.ModeDeplacement))
    g.add((mode, PARIS.nomMode, Literal(ligne.mode_deplacement)))
    if ligne.type_voie:
        voie = uri('type-voie', ligne.type_voie)
        g.add((observation, PARIS.concerneTypeVoie, voie))
        g.add((voie, RDF.type, PARIS.TypeVoie))
        g.add((voie, RDFS.label, Literal(ligne.type_voie)))
print(f'{len(extrait)} observations Silver chargées ; {len(g)} triples.')

In [ ]:
# Requête 1 : quels modes figurent dans cet extrait ?
requete_modes = '''
PREFIX : <http://example.org/comptage-multimodal#>
SELECT DISTINCT ?nom WHERE {
  ?observation a :ObservationComptage ; :concerneMode ?mode .
  ?mode :nomMode ?nom .
} ORDER BY ?nom
'''
resultats_modes = list(g.query(requete_modes))
if not resultats_modes:
    raise ValueError('Aucun mode trouvé dans le graphe.')
for resultat in resultats_modes:
    print(resultat.nom)

In [ ]:
# Requête 2 : quels sites sont représentés ?
requete_sites = '''
PREFIX : <http://example.org/comptage-multimodal#>
SELECT DISTINCT ?identifiant ?nom WHERE {
  ?site a :SiteComptage ; :identifiantSite ?identifiant ; :nomSite ?nom .
} ORDER BY ?nom
'''
resultats_sites = list(g.query(requete_sites))
if not resultats_sites:
    raise ValueError('Aucun site trouvé dans le graphe.')
for resultat in resultats_sites[:10]:
    print(resultat.identifiant, resultat.nom)

In [ ]:
# Requête 3 : quels comptages ont été relevés en janvier 2025 ?
requete_janvier = '''
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?date ?nombre ?nomMode ?identifiant WHERE {
  ?observation a :ObservationComptage ; :dateHeureComptage ?date ;
    :nombreVehicules ?nombre ; :concerneMode ?mode ; :concerneSite ?site .
  ?mode :nomMode ?nomMode .
  ?site :identifiantSite ?identifiant .
  FILTER (?date >= "2025-01-01T00:00:00"^^xsd:dateTime &&
          ?date < "2025-02-01T00:00:00"^^xsd:dateTime)
} ORDER BY ?date LIMIT 20
'''
resultats_janvier = list(g.query(requete_janvier))
if not resultats_janvier:
    raise ValueError('Aucun comptage de janvier 2025 trouvé dans le graphe.')
for resultat in resultats_janvier:
    print(resultat.date, resultat.nombre, resultat.nomMode, resultat.identifiant)

## Export de l'extrait pour Fuseki
Ce fichier contient uniquement l'ontologie et les observations sélectionnées ci-dessus, pas le jeu complet. Il peut être chargé dans Fuseki en suivant le README.

In [ ]:
import tempfile
dossier_rdf = racine / 'data' / 'semantique'
dossier_rdf.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.ttl', prefix='extrait-', dir=dossier_rdf, delete=False) as fichier_rdf:
    fichier_rdf.write(g.serialize(format='turtle'))
print('Extrait RDF exporté :', fichier_rdf.name)